# **Retail Supply Chain Analytics**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.sql("CREATE DATABASE IF NOT EXISTS retail_assessment")
spark.sql("USE retail_assessment")

DataFrame[]

## Part 0 — Prepare Bigger Dataset

In [0]:
products_data = [
(101,"Rice Bag","Groceries","Hyderabad",1200,50),
(102,"Wheat Flour","Groceries","Bengaluru",900,80),
(103,"Sunflower Oil","Groceries","Mumbai",1800,40),
(104,"Milk Pack","Dairy","Chennai",60,200),
(105,"Cheese Block","Dairy","Delhi",450,70),
(106,"Soap","Personal Care","Kolkata",120,300),
(107,"Shampoo","Personal Care","Pune",320,150),
(108,"Toothpaste","Personal Care","Ahmedabad",90,250),
(109,"Notebook","Stationery","Hyderabad",75,500),
(110,"Pen Pack","Stationery","Mumbai",110,400),
(111,"LED TV","Electronics","Delhi",45000,15),
(112,"Refrigerator","Electronics","Chennai",38000,10),
(113,"Washing Machine","Electronics","Bengaluru",29000,12),
(114,"Mobile Phone","Electronics","Hyderabad",25000,35),
(115,"Laptop","Electronics","Pune",62000,18),
(116,"Air Conditioner","Electronics","Mumbai",42000,9),
(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),
(118,"Water Purifier","Home Appliances","Delhi",12000,20),
(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),
(120,"Gas Stove","Home Appliances","Chennai",5500,25)
]

products_columns = ["product_id","product_name","category","warehouse_city","price","stock_quantity"]
products_df = spark.createDataFrame(products_data, products_columns)

suppliers_data = [
(201,"Reddy Traders","Hyderabad","Groceries"),
(202,"Fresh Dairy Ltd","Chennai","Dairy"),
(203,"CarePlus Suppliers","Mumbai","Personal Care"),
(204,"Elite Electronics","Delhi","Electronics"),
(205,"OfficeKart","Bengaluru","Stationery"),
(206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),
(207,"National Grocers","Ahmedabad","Groceries"),
(208,"Smart Electronics","Kolkata","Electronics"),
(209,"Daily Essentials","Hyderabad","Personal Care"),
(210,"Kitchen World","Chennai","Home Appliances")
]

suppliers_columns = ["supplier_id","supplier_name","supplier_city","specialization"]
suppliers_df = spark.createDataFrame(suppliers_data, suppliers_columns)

orders_data = [
(301,101,201,"2024-04-01",20,"Delivered"),
(302,102,201,"2024-04-01",35,"Delivered"),
(303,111,204,"2024-04-02",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Pending"),
(305,115,204,"2024-04-03",3,"Delivered"),
(306,104,202,"2024-04-03",50,"Delivered"),
(307,105,202,"2024-04-04",18,"Cancelled"),
(308,117,206,"2024-04-04",7,"Delivered"),
(309,118,210,"2024-04-05",4,"Pending"),
(310,119,206,"2024-04-05",12,"Delivered"),
(311,120,210,"2024-04-06",6,"Delivered"),
(312,113,204,"2024-04-06",4,"Delivered"),
(313,116,208,"2024-04-07",2,"Pending"),
(314,109,205,"2024-04-07",80,"Delivered"),
(315,110,205,"2024-04-08",120,"Delivered"),
(316,106,203,"2024-04-08",60,"Cancelled"),
(317,107,209,"2024-04-09",25,"Delivered"),
(318,108,203,"2024-04-09",40,"Delivered"),
(319,112,208,"2024-04-10",2,"Pending"),
(320,101,207,"2024-04-10",15,"Delivered")
]

orders_columns = ["order_id","product_id","supplier_id","order_date","quantity","order_status"]
orders_df = spark.createDataFrame(orders_data, orders_columns)

payments_data = [
(401,301,24000,"UPI","Paid"),
(402,302,31500,"Credit Card","Paid"),
(403,303,90000,"Bank Transfer","Paid"),
(404,304,125000,"UPI","Pending"),
(405,305,186000,"Bank Transfer","Paid"),
(406,306,3000,"Cash","Paid"),
(407,307,8100,"UPI","Cancelled"),
(408,308,24500,"Debit Card","Paid"),
(409,309,48000,"UPI","Pending"),
(410,310,33600,"Cash","Paid"),
(411,311,33000,"Credit Card","Paid"),
(412,312,116000,"Bank Transfer","Paid"),
(413,313,84000,"UPI","Pending"),
(414,314,6000,"Cash","Paid"),
(415,315,13200,"UPI","Paid"),
(416,316,7200,"Cash","Cancelled"),
(417,317,8000,"UPI","Paid"),
(418,318,3600,"Debit Card","Paid"),
(419,319,76000,"Bank Transfer","Pending"),
(420,320,18000,"UPI","Paid")
]

payments_columns = ["payment_id","order_id","bill_amount","payment_mode","payment_status"]
payments_df = spark.createDataFrame(payments_data, payments_columns)

## Part 1 — DataFrame Fundamentals

In [0]:
display(products_df)
display(suppliers_df)
display(orders_df)
display(payments_df)

product_id,product_name,category,warehouse_city,price,stock_quantity
101,Rice Bag,Groceries,Hyderabad,1200,50
102,Wheat Flour,Groceries,Bengaluru,900,80
103,Sunflower Oil,Groceries,Mumbai,1800,40
104,Milk Pack,Dairy,Chennai,60,200
105,Cheese Block,Dairy,Delhi,450,70
106,Soap,Personal Care,Kolkata,120,300
107,Shampoo,Personal Care,Pune,320,150
108,Toothpaste,Personal Care,Ahmedabad,90,250
109,Notebook,Stationery,Hyderabad,75,500
110,Pen Pack,Stationery,Mumbai,110,400


supplier_id,supplier_name,supplier_city,specialization
201,Reddy Traders,Hyderabad,Groceries
202,Fresh Dairy Ltd,Chennai,Dairy
203,CarePlus Suppliers,Mumbai,Personal Care
204,Elite Electronics,Delhi,Electronics
205,OfficeKart,Bengaluru,Stationery
206,HomeNeeds Pvt Ltd,Pune,Home Appliances
207,National Grocers,Ahmedabad,Groceries
208,Smart Electronics,Kolkata,Electronics
209,Daily Essentials,Hyderabad,Personal Care
210,Kitchen World,Chennai,Home Appliances


order_id,product_id,supplier_id,order_date,quantity,order_status
301,101,201,2024-04-01,20,Delivered
302,102,201,2024-04-01,35,Delivered
303,111,204,2024-04-02,2,Delivered
304,114,208,2024-04-02,5,Pending
305,115,204,2024-04-03,3,Delivered
306,104,202,2024-04-03,50,Delivered
307,105,202,2024-04-04,18,Cancelled
308,117,206,2024-04-04,7,Delivered
309,118,210,2024-04-05,4,Pending
310,119,206,2024-04-05,12,Delivered


payment_id,order_id,bill_amount,payment_mode,payment_status
401,301,24000,UPI,Paid
402,302,31500,Credit Card,Paid
403,303,90000,Bank Transfer,Paid
404,304,125000,UPI,Pending
405,305,186000,Bank Transfer,Paid
406,306,3000,Cash,Paid
407,307,8100,UPI,Cancelled
408,308,24500,Debit Card,Paid
409,309,48000,UPI,Pending
410,310,33600,Cash,Paid


In [0]:
products_df.printSchema()
suppliers_df.printSchema()
orders_df.printSchema()
payments_df.printSchema()

root
 |-- product_id: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- warehouse_city: string (nullable = true)
 |-- price: long (nullable = true)
 |-- stock_quantity: long (nullable = true)

root
 |-- supplier_id: long (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- supplier_city: string (nullable = true)
 |-- specialization: string (nullable = true)

root
 |-- order_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- supplier_id: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- payment_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- bill_amount: long (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)



In [0]:
print("Products count:", products_df.count())
print("Suppliers count:", suppliers_df.count())
print("Orders count:", orders_df.count())
print("Payments count:", payments_df.count())

Products count: 20
Suppliers count: 10
Orders count: 20
Payments count: 20


In [0]:
display(products_df.limit(10))

product_id,product_name,category,warehouse_city,price,stock_quantity
101,Rice Bag,Groceries,Hyderabad,1200,50
102,Wheat Flour,Groceries,Bengaluru,900,80
103,Sunflower Oil,Groceries,Mumbai,1800,40
104,Milk Pack,Dairy,Chennai,60,200
105,Cheese Block,Dairy,Delhi,450,70
106,Soap,Personal Care,Kolkata,120,300
107,Shampoo,Personal Care,Pune,320,150
108,Toothpaste,Personal Care,Ahmedabad,90,250
109,Notebook,Stationery,Hyderabad,75,500
110,Pen Pack,Stationery,Mumbai,110,400


In [0]:
display(products_df.select("product_name", "category", "stock_quantity"))

product_name,category,stock_quantity
Rice Bag,Groceries,50
Wheat Flour,Groceries,80
Sunflower Oil,Groceries,40
Milk Pack,Dairy,200
Cheese Block,Dairy,70
Soap,Personal Care,300
Shampoo,Personal Care,150
Toothpaste,Personal Care,250
Notebook,Stationery,500
Pen Pack,Stationery,400


In [0]:
display(suppliers_df.filter(col("supplier_city").isin("Hyderabad", "Chennai")))

supplier_id,supplier_name,supplier_city,specialization
201,Reddy Traders,Hyderabad,Groceries
202,Fresh Dairy Ltd,Chennai,Dairy
209,Daily Essentials,Hyderabad,Personal Care
210,Kitchen World,Chennai,Home Appliances


In [0]:
display(orders_df.filter(col("order_status") == "Delivered"))

order_id,product_id,supplier_id,order_date,quantity,order_status
301,101,201,2024-04-01,20,Delivered
302,102,201,2024-04-01,35,Delivered
303,111,204,2024-04-02,2,Delivered
305,115,204,2024-04-03,3,Delivered
306,104,202,2024-04-03,50,Delivered
308,117,206,2024-04-04,7,Delivered
310,119,206,2024-04-05,12,Delivered
311,120,210,2024-04-06,6,Delivered
312,113,204,2024-04-06,4,Delivered
314,109,205,2024-04-07,80,Delivered


In [0]:
display(orders_df.filter(col("order_status") == "Pending"))

order_id,product_id,supplier_id,order_date,quantity,order_status
304,114,208,2024-04-02,5,Pending
309,118,210,2024-04-05,4,Pending
313,116,208,2024-04-07,2,Pending
319,112,208,2024-04-10,2,Pending


In [0]:
display(products_df.filter(col("category") == "Electronics"))

product_id,product_name,category,warehouse_city,price,stock_quantity
111,LED TV,Electronics,Delhi,45000,15
112,Refrigerator,Electronics,Chennai,38000,10
113,Washing Machine,Electronics,Bengaluru,29000,12
114,Mobile Phone,Electronics,Hyderabad,25000,35
115,Laptop,Electronics,Pune,62000,18
116,Air Conditioner,Electronics,Mumbai,42000,9


In [0]:
display(products_df.filter(col("stock_quantity") < 20))

product_id,product_name,category,warehouse_city,price,stock_quantity
111,LED TV,Electronics,Delhi,45000,15
112,Refrigerator,Electronics,Chennai,38000,10
113,Washing Machine,Electronics,Bengaluru,29000,12
115,Laptop,Electronics,Pune,62000,18
116,Air Conditioner,Electronics,Mumbai,42000,9


## Part 2 — DataFrame Transformations

In [0]:
orders_transformed_df = orders_df.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

In [0]:
orders_with_products_df = orders_transformed_df.join(products_df, "product_id", "inner") \
    .withColumn("total_order_value", col("price") * col("quantity"))

In [0]:
products_transformed_df = products_df.withColumn(
    "stock_status",
    when(col("stock_quantity") < 20, "Low Stock")
    .when((col("stock_quantity") >= 20) & (col("stock_quantity") <= 100), "Medium Stock")
    .otherwise("High Stock")
)

In [0]:
orders_transformed_df = orders_transformed_df.withColumn(
    "order_priority",
    when(col("order_status") == "Pending", "High")
    .when(col("order_status") == "Delivered", "Normal")
    .otherwise("Low")
)

In [0]:
products_transformed_df = products_transformed_df.withColumn(
    "expensive_product_flag",
    when(col("price") >= 10000, "Yes").otherwise("No")
)

In [0]:
products_transformed_df = products_transformed_df.withColumn("category", upper(col("category")))

In [0]:
products_transformed_df = products_transformed_df.withColumnRenamed("warehouse_city", "inventory_city")

In [0]:
products_transformed_df = products_transformed_df.withColumn(
    "inventory_value",
    col("price") * col("stock_quantity")
)

In [0]:
products_transformed_df = products_transformed_df.withColumn(
    "low_stock_flag",
    when(col("stock_quantity") < 20, "Yes").otherwise("No")
)

In [0]:
display(products_transformed_df)
display(orders_transformed_df)
display(orders_with_products_df)

product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag
101,Rice Bag,GROCERIES,Hyderabad,1200,50,Medium Stock,No,60000,No
102,Wheat Flour,GROCERIES,Bengaluru,900,80,Medium Stock,No,72000,No
103,Sunflower Oil,GROCERIES,Mumbai,1800,40,Medium Stock,No,72000,No
104,Milk Pack,DAIRY,Chennai,60,200,High Stock,No,12000,No
105,Cheese Block,DAIRY,Delhi,450,70,Medium Stock,No,31500,No
106,Soap,PERSONAL CARE,Kolkata,120,300,High Stock,No,36000,No
107,Shampoo,PERSONAL CARE,Pune,320,150,High Stock,No,48000,No
108,Toothpaste,PERSONAL CARE,Ahmedabad,90,250,High Stock,No,22500,No
109,Notebook,STATIONERY,Hyderabad,75,500,High Stock,No,37500,No
110,Pen Pack,STATIONERY,Mumbai,110,400,High Stock,No,44000,No


order_id,product_id,supplier_id,order_date,quantity,order_status,order_priority
301,101,201,2024-04-01,20,Delivered,Normal
302,102,201,2024-04-01,35,Delivered,Normal
303,111,204,2024-04-02,2,Delivered,Normal
304,114,208,2024-04-02,5,Pending,High
305,115,204,2024-04-03,3,Delivered,Normal
306,104,202,2024-04-03,50,Delivered,Normal
307,105,202,2024-04-04,18,Cancelled,Low
308,117,206,2024-04-04,7,Delivered,Normal
309,118,210,2024-04-05,4,Pending,High
310,119,206,2024-04-05,12,Delivered,Normal


product_id,order_id,supplier_id,order_date,quantity,order_status,product_name,category,warehouse_city,price,stock_quantity,total_order_value
101,320,207,2024-04-10,15,Delivered,Rice Bag,Groceries,Hyderabad,1200,50,18000
102,302,201,2024-04-01,35,Delivered,Wheat Flour,Groceries,Bengaluru,900,80,31500
101,301,201,2024-04-01,20,Delivered,Rice Bag,Groceries,Hyderabad,1200,50,24000
104,306,202,2024-04-03,50,Delivered,Milk Pack,Dairy,Chennai,60,200,3000
105,307,202,2024-04-04,18,Cancelled,Cheese Block,Dairy,Delhi,450,70,8100
106,316,203,2024-04-08,60,Cancelled,Soap,Personal Care,Kolkata,120,300,7200
107,317,209,2024-04-09,25,Delivered,Shampoo,Personal Care,Pune,320,150,8000
108,318,203,2024-04-09,40,Delivered,Toothpaste,Personal Care,Ahmedabad,90,250,3600
109,314,205,2024-04-07,80,Delivered,Notebook,Stationery,Hyderabad,75,500,6000
110,315,205,2024-04-08,120,Delivered,Pen Pack,Stationery,Mumbai,110,400,13200


## Part 3 — Joins

In [0]:
products_orders_df = products_transformed_df.join(orders_transformed_df, "product_id", "inner")

In [0]:
orders_suppliers_df = orders_transformed_df.join(suppliers_df, "supplier_id", "inner")

In [0]:
orders_payments_df = orders_transformed_df.join(payments_df, "order_id", "inner")

In [0]:
final_joined_df = products_transformed_df \
    .join(orders_transformed_df, "product_id", "inner") \
    .join(suppliers_df, "supplier_id", "inner") \
    .join(payments_df, "order_id", "inner") \
    .withColumn("total_order_value", col("price") * col("quantity"))

display(final_joined_df)

order_id,supplier_id,product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag,order_date,quantity,order_status,order_priority,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status,total_order_value
301,201,101,Rice Bag,GROCERIES,Hyderabad,1200,50,Medium Stock,No,60000,No,2024-04-01,20,Delivered,Normal,Reddy Traders,Hyderabad,Groceries,401,24000,UPI,Paid,24000
302,201,102,Wheat Flour,GROCERIES,Bengaluru,900,80,Medium Stock,No,72000,No,2024-04-01,35,Delivered,Normal,Reddy Traders,Hyderabad,Groceries,402,31500,Credit Card,Paid,31500
303,204,111,LED TV,ELECTRONICS,Delhi,45000,15,Low Stock,Yes,675000,Yes,2024-04-02,2,Delivered,Normal,Elite Electronics,Delhi,Electronics,403,90000,Bank Transfer,Paid,90000
304,208,114,Mobile Phone,ELECTRONICS,Hyderabad,25000,35,Medium Stock,Yes,875000,No,2024-04-02,5,Pending,High,Smart Electronics,Kolkata,Electronics,404,125000,UPI,Pending,125000
305,204,115,Laptop,ELECTRONICS,Pune,62000,18,Low Stock,Yes,1116000,Yes,2024-04-03,3,Delivered,Normal,Elite Electronics,Delhi,Electronics,405,186000,Bank Transfer,Paid,186000
306,202,104,Milk Pack,DAIRY,Chennai,60,200,High Stock,No,12000,No,2024-04-03,50,Delivered,Normal,Fresh Dairy Ltd,Chennai,Dairy,406,3000,Cash,Paid,3000
307,202,105,Cheese Block,DAIRY,Delhi,450,70,Medium Stock,No,31500,No,2024-04-04,18,Cancelled,Low,Fresh Dairy Ltd,Chennai,Dairy,407,8100,UPI,Cancelled,8100
308,206,117,Mixer Grinder,HOME APPLIANCES,Kolkata,3500,45,Medium Stock,No,157500,No,2024-04-04,7,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,408,24500,Debit Card,Paid,24500
309,210,118,Water Purifier,HOME APPLIANCES,Delhi,12000,20,Medium Stock,Yes,240000,No,2024-04-05,4,Pending,High,Kitchen World,Chennai,Home Appliances,409,48000,UPI,Pending,48000
310,206,119,Ceiling Fan,HOME APPLIANCES,Ahmedabad,2800,60,Medium Stock,No,168000,No,2024-04-05,12,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,410,33600,Cash,Paid,33600


In [0]:
display(final_joined_df.select("product_name", "supplier_name", "quantity", "bill_amount"))

product_name,supplier_name,quantity,bill_amount
Rice Bag,Reddy Traders,20,24000
Wheat Flour,Reddy Traders,35,31500
LED TV,Elite Electronics,2,90000
Mobile Phone,Smart Electronics,5,125000
Laptop,Elite Electronics,3,186000
Milk Pack,Fresh Dairy Ltd,50,3000
Cheese Block,Fresh Dairy Ltd,18,8100
Mixer Grinder,HomeNeeds Pvt Ltd,7,24500
Water Purifier,Kitchen World,4,48000
Ceiling Fan,HomeNeeds Pvt Ltd,12,33600


In [0]:
display(final_joined_df.filter(col("inventory_city") != col("supplier_city")) \
    .select("supplier_name", "supplier_city", "product_name", "inventory_city"))

supplier_name,supplier_city,product_name,inventory_city
Reddy Traders,Hyderabad,Wheat Flour,Bengaluru
Smart Electronics,Kolkata,Mobile Phone,Hyderabad
Elite Electronics,Delhi,Laptop,Pune
Fresh Dairy Ltd,Chennai,Cheese Block,Delhi
HomeNeeds Pvt Ltd,Pune,Mixer Grinder,Kolkata
Kitchen World,Chennai,Water Purifier,Delhi
HomeNeeds Pvt Ltd,Pune,Ceiling Fan,Ahmedabad
Elite Electronics,Delhi,Washing Machine,Bengaluru
Smart Electronics,Kolkata,Air Conditioner,Mumbai
OfficeKart,Bengaluru,Notebook,Hyderabad


In [0]:
display(final_joined_df.filter((col("order_status") == "Delivered") & (col("payment_status") == "Paid")))

order_id,supplier_id,product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag,order_date,quantity,order_status,order_priority,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status,total_order_value
301,201,101,Rice Bag,GROCERIES,Hyderabad,1200,50,Medium Stock,No,60000,No,2024-04-01,20,Delivered,Normal,Reddy Traders,Hyderabad,Groceries,401,24000,UPI,Paid,24000
302,201,102,Wheat Flour,GROCERIES,Bengaluru,900,80,Medium Stock,No,72000,No,2024-04-01,35,Delivered,Normal,Reddy Traders,Hyderabad,Groceries,402,31500,Credit Card,Paid,31500
303,204,111,LED TV,ELECTRONICS,Delhi,45000,15,Low Stock,Yes,675000,Yes,2024-04-02,2,Delivered,Normal,Elite Electronics,Delhi,Electronics,403,90000,Bank Transfer,Paid,90000
305,204,115,Laptop,ELECTRONICS,Pune,62000,18,Low Stock,Yes,1116000,Yes,2024-04-03,3,Delivered,Normal,Elite Electronics,Delhi,Electronics,405,186000,Bank Transfer,Paid,186000
306,202,104,Milk Pack,DAIRY,Chennai,60,200,High Stock,No,12000,No,2024-04-03,50,Delivered,Normal,Fresh Dairy Ltd,Chennai,Dairy,406,3000,Cash,Paid,3000
308,206,117,Mixer Grinder,HOME APPLIANCES,Kolkata,3500,45,Medium Stock,No,157500,No,2024-04-04,7,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,408,24500,Debit Card,Paid,24500
310,206,119,Ceiling Fan,HOME APPLIANCES,Ahmedabad,2800,60,Medium Stock,No,168000,No,2024-04-05,12,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,410,33600,Cash,Paid,33600
311,210,120,Gas Stove,HOME APPLIANCES,Chennai,5500,25,Medium Stock,No,137500,No,2024-04-06,6,Delivered,Normal,Kitchen World,Chennai,Home Appliances,411,33000,Credit Card,Paid,33000
312,204,113,Washing Machine,ELECTRONICS,Bengaluru,29000,12,Low Stock,Yes,348000,Yes,2024-04-06,4,Delivered,Normal,Elite Electronics,Delhi,Electronics,412,116000,Bank Transfer,Paid,116000
314,205,109,Notebook,STATIONERY,Hyderabad,75,500,High Stock,No,37500,No,2024-04-07,80,Delivered,Normal,OfficeKart,Bengaluru,Stationery,414,6000,Cash,Paid,6000


In [0]:
display(final_joined_df.filter((col("order_status") == "Pending") & (col("payment_status") == "Pending")))

order_id,supplier_id,product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag,order_date,quantity,order_status,order_priority,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status,total_order_value
304,208,114,Mobile Phone,ELECTRONICS,Hyderabad,25000,35,Medium Stock,Yes,875000,No,2024-04-02,5,Pending,High,Smart Electronics,Kolkata,Electronics,404,125000,UPI,Pending,125000
309,210,118,Water Purifier,HOME APPLIANCES,Delhi,12000,20,Medium Stock,Yes,240000,No,2024-04-05,4,Pending,High,Kitchen World,Chennai,Home Appliances,409,48000,UPI,Pending,48000
313,208,116,Air Conditioner,ELECTRONICS,Mumbai,42000,9,Low Stock,Yes,378000,Yes,2024-04-07,2,Pending,High,Smart Electronics,Kolkata,Electronics,413,84000,UPI,Pending,84000
319,208,112,Refrigerator,ELECTRONICS,Chennai,38000,10,Low Stock,Yes,380000,Yes,2024-04-10,2,Pending,High,Smart Electronics,Kolkata,Electronics,419,76000,Bank Transfer,Pending,76000


In [0]:
display(final_joined_df.filter((col("order_status") == "Cancelled") & (col("payment_status") == "Cancelled")))

order_id,supplier_id,product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag,order_date,quantity,order_status,order_priority,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status,total_order_value
307,202,105,Cheese Block,DAIRY,Delhi,450,70,Medium Stock,No,31500,No,2024-04-04,18,Cancelled,Low,Fresh Dairy Ltd,Chennai,Dairy,407,8100,UPI,Cancelled,8100
316,203,106,Soap,PERSONAL CARE,Kolkata,120,300,High Stock,No,36000,No,2024-04-08,60,Cancelled,Low,CarePlus Suppliers,Mumbai,Personal Care,416,7200,Cash,Cancelled,7200


In [0]:
display(final_joined_df.groupBy("product_id", "product_name").count().filter(col("count") > 1))

product_id,product_name,count
101,Rice Bag,2


## Part 4 — Aggregations

In [0]:
display(products_transformed_df.groupBy("category").count())

display(orders_transformed_df.groupBy("order_status").count())

display(suppliers_df.groupBy("supplier_city").count())

display(payments_df.agg(sum("bill_amount").alias("total_revenue")))

display(payments_df.agg(avg("bill_amount").alias("average_bill_amount")))

display(final_joined_df.groupBy("category").agg(sum("bill_amount").alias("total_revenue")))

display(final_joined_df.groupBy("supplier_name").agg(sum("bill_amount").alias("total_revenue")))

display(final_joined_df.groupBy("inventory_city").agg(sum("bill_amount").alias("total_revenue")))

display(final_joined_df.groupBy("product_name").agg(sum("quantity").alias("total_quantity_sold")).orderBy(col("total_quantity_sold").desc()))

display(final_joined_df.groupBy("product_name").agg(sum("quantity").alias("total_quantity_sold")).orderBy(col("total_quantity_sold").asc()))

category,count
GROCERIES,3
DAIRY,2
PERSONAL CARE,3
STATIONERY,2
ELECTRONICS,6
HOME APPLIANCES,4


order_status,count
Delivered,14
Pending,4
Cancelled,2


supplier_city,count
Hyderabad,2
Chennai,2
Mumbai,1
Delhi,1
Bengaluru,1
Pune,1
Ahmedabad,1
Kolkata,1


total_revenue
938700


average_bill_amount
46935.0


category,total_revenue
GROCERIES,73500
ELECTRONICS,677000
DAIRY,11100
HOME APPLIANCES,139100
STATIONERY,19200
PERSONAL CARE,18800


supplier_name,total_revenue
Reddy Traders,55500
Smart Electronics,285000
Elite Electronics,392000
Fresh Dairy Ltd,11100
HomeNeeds Pvt Ltd,58100
Kitchen World,81000
OfficeKart,19200
CarePlus Suppliers,10800
Daily Essentials,8000
National Grocers,18000


inventory_city,total_revenue
Hyderabad,173000
Bengaluru,147500
Delhi,146100
Pune,194000
Chennai,112000
Ahmedabad,37200
Kolkata,31700
Mumbai,97200


product_name,total_quantity_sold
Pen Pack,120
Notebook,80
Soap,60
Milk Pack,50
Toothpaste,40
Rice Bag,35
Wheat Flour,35
Shampoo,25
Cheese Block,18
Ceiling Fan,12


product_name,total_quantity_sold
LED TV,2
Air Conditioner,2
Refrigerator,2
Laptop,3
Water Purifier,4
Washing Machine,4
Mobile Phone,5
Gas Stove,6
Mixer Grinder,7
Ceiling Fan,12


## Part 5 — Spark SQL

In [0]:
products_transformed_df.createOrReplaceTempView("products")
suppliers_df.createOrReplaceTempView("suppliers")
orders_transformed_df.createOrReplaceTempView("orders")
payments_df.createOrReplaceTempView("payments")
final_joined_df.createOrReplaceTempView("retail_final")

In [0]:
display(spark.sql("SELECT * FROM products"))

product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag
101,Rice Bag,GROCERIES,Hyderabad,1200,50,Medium Stock,No,60000,No
102,Wheat Flour,GROCERIES,Bengaluru,900,80,Medium Stock,No,72000,No
103,Sunflower Oil,GROCERIES,Mumbai,1800,40,Medium Stock,No,72000,No
104,Milk Pack,DAIRY,Chennai,60,200,High Stock,No,12000,No
105,Cheese Block,DAIRY,Delhi,450,70,Medium Stock,No,31500,No
106,Soap,PERSONAL CARE,Kolkata,120,300,High Stock,No,36000,No
107,Shampoo,PERSONAL CARE,Pune,320,150,High Stock,No,48000,No
108,Toothpaste,PERSONAL CARE,Ahmedabad,90,250,High Stock,No,22500,No
109,Notebook,STATIONERY,Hyderabad,75,500,High Stock,No,37500,No
110,Pen Pack,STATIONERY,Mumbai,110,400,High Stock,No,44000,No


In [0]:
display(spark.sql("""
SELECT *
FROM retail_final
WHERE category = 'ELECTRONICS'
"""))

order_id,supplier_id,product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag,order_date,quantity,order_status,order_priority,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status,total_order_value
303,204,111,LED TV,ELECTRONICS,Delhi,45000,15,Low Stock,Yes,675000,Yes,2024-04-02,2,Delivered,Normal,Elite Electronics,Delhi,Electronics,403,90000,Bank Transfer,Paid,90000
304,208,114,Mobile Phone,ELECTRONICS,Hyderabad,25000,35,Medium Stock,Yes,875000,No,2024-04-02,5,Pending,High,Smart Electronics,Kolkata,Electronics,404,125000,UPI,Pending,125000
305,204,115,Laptop,ELECTRONICS,Pune,62000,18,Low Stock,Yes,1116000,Yes,2024-04-03,3,Delivered,Normal,Elite Electronics,Delhi,Electronics,405,186000,Bank Transfer,Paid,186000
312,204,113,Washing Machine,ELECTRONICS,Bengaluru,29000,12,Low Stock,Yes,348000,Yes,2024-04-06,4,Delivered,Normal,Elite Electronics,Delhi,Electronics,412,116000,Bank Transfer,Paid,116000
313,208,116,Air Conditioner,ELECTRONICS,Mumbai,42000,9,Low Stock,Yes,378000,Yes,2024-04-07,2,Pending,High,Smart Electronics,Kolkata,Electronics,413,84000,UPI,Pending,84000
319,208,112,Refrigerator,ELECTRONICS,Chennai,38000,10,Low Stock,Yes,380000,Yes,2024-04-10,2,Pending,High,Smart Electronics,Kolkata,Electronics,419,76000,Bank Transfer,Pending,76000


In [0]:
display(spark.sql("""
SELECT category, SUM(bill_amount) AS total_revenue
FROM retail_final
GROUP BY category
"""))

category,total_revenue
GROCERIES,73500
ELECTRONICS,677000
DAIRY,11100
HOME APPLIANCES,139100
STATIONERY,19200
PERSONAL CARE,18800


In [0]:
display(spark.sql("""
SELECT supplier_name, SUM(bill_amount) AS total_revenue
FROM retail_final
GROUP BY supplier_name
"""))

supplier_name,total_revenue
Reddy Traders,55500
Smart Electronics,285000
Elite Electronics,392000
Fresh Dairy Ltd,11100
HomeNeeds Pvt Ltd,58100
Kitchen World,81000
OfficeKart,19200
CarePlus Suppliers,10800
Daily Essentials,8000
National Grocers,18000


In [0]:
display(spark.sql("""
SELECT order_id, product_name, supplier_name, bill_amount
FROM retail_final
ORDER BY bill_amount DESC
LIMIT 5
"""))

order_id,product_name,supplier_name,bill_amount
305,Laptop,Elite Electronics,186000
304,Mobile Phone,Smart Electronics,125000
312,Washing Machine,Elite Electronics,116000
303,LED TV,Elite Electronics,90000
313,Air Conditioner,Smart Electronics,84000


In [0]:
display(spark.sql("""
SELECT supplier_name, COUNT(order_id) AS total_orders
FROM retail_final
GROUP BY supplier_name
"""))

supplier_name,total_orders
Reddy Traders,2
Smart Electronics,3
Elite Electronics,3
Fresh Dairy Ltd,2
HomeNeeds Pvt Ltd,2
Kitchen World,2
OfficeKart,2
CarePlus Suppliers,2
Daily Essentials,1
National Grocers,1


In [0]:
display(spark.sql("""
SELECT category, COUNT(order_id) AS total_orders
FROM retail_final
GROUP BY category
"""))

category,total_orders
GROCERIES,3
ELECTRONICS,6
DAIRY,2
HOME APPLIANCES,4
STATIONERY,2
PERSONAL CARE,3


In [0]:
display(spark.sql("""
SELECT payment_mode, AVG(bill_amount) AS average_payment
FROM retail_final
GROUP BY payment_mode
"""))

payment_mode,average_payment
Credit Card,32250.0
UPI,41037.5
Bank Transfer,117000.0
Cash,12450.0
Debit Card,14050.0


In [0]:
display(spark.sql("""
SELECT product_name, SUM(bill_amount) AS total_revenue
FROM retail_final
GROUP BY product_name
HAVING SUM(bill_amount) > 100000
"""))

product_name,total_revenue
Laptop,186000
Mobile Phone,125000
Washing Machine,116000


## Part 6 — Window Functions

In [0]:
product_revenue_df = final_joined_df.groupBy("category", "product_name") \
    .agg(sum("bill_amount").alias("total_revenue"))

window_category = Window.partitionBy("category").orderBy(col("total_revenue").desc())

display(product_revenue_df.withColumn("rank_within_category", rank().over(window_category)))

category,product_name,total_revenue,rank_within_category
DAIRY,Cheese Block,8100,1
DAIRY,Milk Pack,3000,2
ELECTRONICS,Laptop,186000,1
ELECTRONICS,Mobile Phone,125000,2
ELECTRONICS,Washing Machine,116000,3
ELECTRONICS,LED TV,90000,4
ELECTRONICS,Air Conditioner,84000,5
ELECTRONICS,Refrigerator,76000,6
GROCERIES,Rice Bag,42000,1
GROCERIES,Wheat Flour,31500,2


In [0]:
supplier_city_revenue_df = final_joined_df.groupBy("supplier_city", "supplier_name") \
    .agg(sum("bill_amount").alias("total_revenue"))

window_supplier_city = Window.partitionBy("supplier_city").orderBy(col("total_revenue").desc())

display(supplier_city_revenue_df.withColumn("rank_within_city", rank().over(window_supplier_city)))

supplier_city,supplier_name,total_revenue,rank_within_city
Ahmedabad,National Grocers,18000,1
Bengaluru,OfficeKart,19200,1
Chennai,Kitchen World,81000,1
Chennai,Fresh Dairy Ltd,11100,2
Delhi,Elite Electronics,392000,1
Hyderabad,Reddy Traders,55500,1
Hyderabad,Daily Essentials,8000,2
Kolkata,Smart Electronics,285000,1
Mumbai,CarePlus Suppliers,10800,1
Pune,HomeNeeds Pvt Ltd,58100,1


In [0]:
display(product_revenue_df.withColumn("row_num", row_number().over(window_category)).filter(col("row_num") == 1))

category,product_name,total_revenue,row_num
DAIRY,Cheese Block,8100,1
ELECTRONICS,Laptop,186000,1
GROCERIES,Rice Bag,42000,1
HOME APPLIANCES,Water Purifier,48000,1
PERSONAL CARE,Shampoo,8000,1
STATIONERY,Pen Pack,13200,1


In [0]:
supplier_bill_df = final_joined_df.groupBy("supplier_name").agg(sum("bill_amount").alias("total_bill"))

In [0]:
window_supplier_bill = Window.orderBy(col("total_bill").desc())
display(supplier_bill_df.withColumn("dense_rank", dense_rank().over(window_supplier_bill)))

/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:968: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


supplier_name,total_bill,dense_rank
Elite Electronics,392000,1
Smart Electronics,285000,2
Kitchen World,81000,3
HomeNeeds Pvt Ltd,58100,4
Reddy Traders,55500,5
OfficeKart,19200,6
National Grocers,18000,7
Fresh Dairy Ltd,11100,8
CarePlus Suppliers,10800,9
Daily Essentials,8000,10


In [0]:
display(supplier_bill_df.withColumn("rank", rank().over(window_supplier_bill)).filter(col("rank") <= 2))

/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:968: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


supplier_name,total_bill,rank
Elite Electronics,392000,1
Smart Electronics,285000,2


In [0]:
daily_revenue_df = final_joined_df.groupBy("order_date").agg(sum("bill_amount").alias("daily_revenue"))

window_date = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

display(daily_revenue_df.withColumn("running_total_revenue", sum("daily_revenue").over(window_date)))

/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:968: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


order_date,daily_revenue,running_total_revenue
2024-04-01,55500,55500
2024-04-02,215000,270500
2024-04-03,189000,459500
2024-04-04,32600,492100
2024-04-05,81600,573700
2024-04-06,149000,722700
2024-04-07,90000,812700
2024-04-08,20400,833100
2024-04-09,11600,844700
2024-04-10,94000,938700


In [0]:
supplier_date_revenue_df = final_joined_df.groupBy("supplier_name", "order_date") \
    .agg(sum("bill_amount").alias("daily_supplier_revenue"))

window_supplier_date = Window.partitionBy("supplier_name").orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

display(supplier_date_revenue_df.withColumn("running_revenue_by_supplier", sum("daily_supplier_revenue").over(window_supplier_date)))

supplier_name,order_date,daily_supplier_revenue,running_revenue_by_supplier
CarePlus Suppliers,2024-04-08,7200,7200
CarePlus Suppliers,2024-04-09,3600,10800
Daily Essentials,2024-04-09,8000,8000
Elite Electronics,2024-04-02,90000,90000
Elite Electronics,2024-04-03,186000,276000
Elite Electronics,2024-04-06,116000,392000
Fresh Dairy Ltd,2024-04-03,3000,3000
Fresh Dairy Ltd,2024-04-04,8100,11100
HomeNeeds Pvt Ltd,2024-04-04,24500,24500
HomeNeeds Pvt Ltd,2024-04-05,33600,58100


In [0]:
city_revenue_df = final_joined_df.groupBy("inventory_city").agg(sum("bill_amount").alias("total_revenue"))
window_city_revenue = Window.orderBy(col("total_revenue").desc())
display(city_revenue_df.withColumn("city_rank", rank().over(window_city_revenue)))

/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:968: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


inventory_city,total_revenue,city_rank
Pune,194000,1
Hyderabad,173000,2
Bengaluru,147500,3
Delhi,146100,4
Chennai,112000,5
Mumbai,97200,6
Ahmedabad,37200,7
Kolkata,31700,8


In [0]:
category_revenue_df = final_joined_df.groupBy("category").agg(sum("bill_amount").alias("total_revenue"))
window_category_revenue = Window.orderBy(col("total_revenue").desc())
display(category_revenue_df.withColumn("category_rank", rank().over(window_category_revenue)))

/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:968: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


category,total_revenue,category_rank
ELECTRONICS,677000,1
HOME APPLIANCES,139100,2
GROCERIES,73500,3
STATIONERY,19200,4
PERSONAL CARE,18800,5
DAIRY,11100,6


In [0]:
window_payment_mode = Window.partitionBy("payment_mode").orderBy(col("bill_amount").desc())

display(final_joined_df.withColumn("row_num", row_number().over(window_payment_mode)) \
    .filter(col("row_num") == 1) \
    .select("payment_mode", "order_id", "product_name", "supplier_name", "bill_amount"))

payment_mode,order_id,product_name,supplier_name,bill_amount
Bank Transfer,305,Laptop,Elite Electronics,186000
Cash,310,Ceiling Fan,HomeNeeds Pvt Ltd,33600
Credit Card,311,Gas Stove,Kitchen World,33000
Debit Card,308,Mixer Grinder,HomeNeeds Pvt Ltd,24500
UPI,304,Mobile Phone,Smart Electronics,125000


## Part 7 — Delta Lake Core

In [0]:
spark.sql("DROP TABLE IF EXISTS retail_final_delta")
final_joined_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_final_delta")
display(spark.table("retail_final_delta"))

order_id,supplier_id,product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag,order_date,quantity,order_status,order_priority,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status,total_order_value
313,208,116,Air Conditioner,ELECTRONICS,Mumbai,42000,9,Low Stock,Yes,378000,Yes,2024-04-07,2,Pending,High,Smart Electronics,Kolkata,Electronics,413,84000,UPI,Pending,84000
314,205,109,Notebook,STATIONERY,Hyderabad,75,500,High Stock,No,37500,No,2024-04-07,80,Delivered,Normal,OfficeKart,Bengaluru,Stationery,414,6000,Cash,Paid,6000
315,205,110,Pen Pack,STATIONERY,Mumbai,110,400,High Stock,No,44000,No,2024-04-08,120,Delivered,Normal,OfficeKart,Bengaluru,Stationery,415,13200,UPI,Paid,13200
308,206,117,Mixer Grinder,HOME APPLIANCES,Kolkata,3500,45,Medium Stock,No,157500,No,2024-04-04,7,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,408,24500,Debit Card,Paid,24500
309,210,118,Water Purifier,HOME APPLIANCES,Delhi,12000,20,Medium Stock,Yes,240000,No,2024-04-05,4,Pending,High,Kitchen World,Chennai,Home Appliances,409,48000,UPI,Pending,48000
310,206,119,Ceiling Fan,HOME APPLIANCES,Ahmedabad,2800,60,Medium Stock,No,168000,No,2024-04-05,12,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,410,33600,Cash,Paid,33600
303,204,111,LED TV,ELECTRONICS,Delhi,45000,15,Low Stock,Yes,675000,Yes,2024-04-02,2,Delivered,Normal,Elite Electronics,Delhi,Electronics,403,90000,Bank Transfer,Paid,90000
304,208,114,Mobile Phone,ELECTRONICS,Hyderabad,25000,35,Medium Stock,Yes,875000,No,2024-04-02,5,Pending,High,Smart Electronics,Kolkata,Electronics,404,125000,UPI,Pending,125000
305,204,115,Laptop,ELECTRONICS,Pune,62000,18,Low Stock,Yes,1116000,Yes,2024-04-03,3,Delivered,Normal,Elite Electronics,Delhi,Electronics,405,186000,Bank Transfer,Paid,186000
318,203,108,Toothpaste,PERSONAL CARE,Ahmedabad,90,250,High Stock,No,22500,No,2024-04-09,40,Delivered,Normal,CarePlus Suppliers,Mumbai,Personal Care,418,3600,Debit Card,Paid,3600


In [0]:
new_orders_data = [
    (321, 114, 204, "2024-04-11", 3, "Delivered", 421, 75000, "UPI", "Paid"),
    (322, 118, 210, "2024-04-11", 2, "Pending", 422, 24000, "Cash", "Pending")
]

new_orders_cols = [
    "order_id", "product_id", "supplier_id", "order_date", "quantity",
    "order_status", "payment_id", "bill_amount", "payment_mode", "payment_status"
]

new_orders_base_df = spark.createDataFrame(new_orders_data, new_orders_cols) \
    .withColumn("order_date", to_date("order_date"))

new_joined_rows_df = products_transformed_df \
    .join(new_orders_base_df, "product_id") \
    .join(suppliers_df, "supplier_id") \
    .withColumn("total_order_value", col("price") * col("quantity")) \
    .withColumn("order_priority",
        when(col("order_status") == "Pending", "High")
        .when(col("order_status") == "Delivered", "Normal")
        .otherwise("Low")) \
    .select(final_joined_df.columns)

new_joined_rows_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("retail_final_delta")

display(spark.sql("SELECT order_id, product_name, supplier_name, order_status, bill_amount, payment_status FROM retail_final_delta WHERE order_id IN (321, 322)"))

order_id,product_name,supplier_name,order_status,bill_amount,payment_status
322,Water Purifier,Kitchen World,Pending,24000,Pending
321,Mobile Phone,Elite Electronics,Delivered,75000,Paid


In [0]:
spark.sql("UPDATE retail_final_delta SET order_status = 'Delivered' WHERE order_id = 322")
display(spark.sql("SELECT order_id, product_name, order_status FROM retail_final_delta WHERE order_id = 322"))

order_id,product_name,order_status
322,Water Purifier,Delivered


In [0]:
spark.sql("UPDATE retail_final_delta SET payment_status = 'Paid' WHERE payment_id = 422")
display(spark.sql("SELECT payment_id, order_id, payment_status FROM retail_final_delta WHERE payment_id = 422"))

payment_id,order_id,payment_status
422,322,Paid


In [0]:
spark.sql("DELETE FROM retail_final_delta WHERE order_status = 'Cancelled'")
display(spark.sql("SELECT order_id, order_status FROM retail_final_delta WHERE order_status = 'Cancelled'"))

order_id,order_status


In [0]:
spark.sql("DROP TABLE IF EXISTS clean_orders_delta")

spark.sql("""
CREATE TABLE clean_orders_delta
USING DELTA
AS
SELECT *
FROM retail_final_delta
WHERE order_status = 'Delivered'
AND payment_status = 'Paid'
""")

display(spark.table("clean_orders_delta"))


order_id,supplier_id,product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag,order_date,quantity,order_status,order_priority,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status,total_order_value
301,201,101,Rice Bag,GROCERIES,Hyderabad,1200,50,Medium Stock,No,60000,No,2024-04-01,20,Delivered,Normal,Reddy Traders,Hyderabad,Groceries,401,24000,UPI,Paid,24000
302,201,102,Wheat Flour,GROCERIES,Bengaluru,900,80,Medium Stock,No,72000,No,2024-04-01,35,Delivered,Normal,Reddy Traders,Hyderabad,Groceries,402,31500,Credit Card,Paid,31500
303,204,111,LED TV,ELECTRONICS,Delhi,45000,15,Low Stock,Yes,675000,Yes,2024-04-02,2,Delivered,Normal,Elite Electronics,Delhi,Electronics,403,90000,Bank Transfer,Paid,90000
305,204,115,Laptop,ELECTRONICS,Pune,62000,18,Low Stock,Yes,1116000,Yes,2024-04-03,3,Delivered,Normal,Elite Electronics,Delhi,Electronics,405,186000,Bank Transfer,Paid,186000
306,202,104,Milk Pack,DAIRY,Chennai,60,200,High Stock,No,12000,No,2024-04-03,50,Delivered,Normal,Fresh Dairy Ltd,Chennai,Dairy,406,3000,Cash,Paid,3000
308,206,117,Mixer Grinder,HOME APPLIANCES,Kolkata,3500,45,Medium Stock,No,157500,No,2024-04-04,7,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,408,24500,Debit Card,Paid,24500
310,206,119,Ceiling Fan,HOME APPLIANCES,Ahmedabad,2800,60,Medium Stock,No,168000,No,2024-04-05,12,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,410,33600,Cash,Paid,33600
311,210,120,Gas Stove,HOME APPLIANCES,Chennai,5500,25,Medium Stock,No,137500,No,2024-04-06,6,Delivered,Normal,Kitchen World,Chennai,Home Appliances,411,33000,Credit Card,Paid,33000
312,204,113,Washing Machine,ELECTRONICS,Bengaluru,29000,12,Low Stock,Yes,348000,Yes,2024-04-06,4,Delivered,Normal,Elite Electronics,Delhi,Electronics,412,116000,Bank Transfer,Paid,116000
314,205,109,Notebook,STATIONERY,Hyderabad,75,500,High Stock,No,37500,No,2024-04-07,80,Delivered,Normal,OfficeKart,Bengaluru,Stationery,414,6000,Cash,Paid,6000


In [0]:
display(spark.sql("DESCRIBE HISTORY retail_final_delta"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-05-06T10:45:21.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2392048196024932),474a2517-82ae-43f0-b505-1c218b9bea88,0506-100524-iqf6z1vx-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 10, numRemovedBytes -> 64644, p25FileSize -> 8265, numDeletionVectorsRemoved -> 2, minFileSize -> 8265, numAddedFiles -> 1, maxFileSize -> 8265, p75FileSize -> 8265, p50FileSize -> 8265, numAddedBytes -> 8265)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-06T10:45:20.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(order_status#32731 = Cancelled)""])",null,List(2392048196024932),474a2517-82ae-43f0-b505-1c218b9bea88,0506-100524-iqf6z1vx-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 2, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1284, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 958, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 326)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-06T10:45:02.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(payment_id#31586L = 422)""])",null,List(2392048196024932),a748466b-5f77-4de5-9008-eec730297a1b,0506-100524-iqf6z1vx-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 6059, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1927, numDeletionVectorsUpdated -> 0, scanTimeMs -> 928, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 6043, rewriteTimeMs -> 998)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-06T10:44:36.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(order_id#30417L = 322)""])",null,List(2392048196024932),da6ddca6-907a-4c96-8592-de1bad707336,0506-100524-iqf6z1vx-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 6025, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2042, numDeletionVectorsUpdated -> 0, scanTimeMs -> 983, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 6059, rewriteTimeMs -> 1058)",null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-05-06T10:44:15.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2392048196024932),b18a06e2-892d-4f7e-8149-5aaf79123245,0506-100524-iqf6z1vx-v2n,0,WriteSerializable,true,"Map(numFiles -> 2, numOutputRows -> 2, numOutputBytes -> 12031)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-06T10:43:25.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2392048196024932),66d921e0-d1c1-4760-8c5b-9447778d0ed2,0506-100524-iqf6z1vx-v2n,null,WriteSerializable,false,"Map(numFiles -> 8, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 20, numOutputBytes -> 52595)",null,Databricks-Runtime/18.1.x-photon-scala2.13


In [0]:
older_retail_df = spark.read.format("delta").option("versionAsOf", 0).table("retail_final_delta")
display(older_retail_df)

order_id,supplier_id,product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag,order_date,quantity,order_status,order_priority,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status,total_order_value
313,208,116,Air Conditioner,ELECTRONICS,Mumbai,42000,9,Low Stock,Yes,378000,Yes,2024-04-07,2,Pending,High,Smart Electronics,Kolkata,Electronics,413,84000,UPI,Pending,84000
314,205,109,Notebook,STATIONERY,Hyderabad,75,500,High Stock,No,37500,No,2024-04-07,80,Delivered,Normal,OfficeKart,Bengaluru,Stationery,414,6000,Cash,Paid,6000
315,205,110,Pen Pack,STATIONERY,Mumbai,110,400,High Stock,No,44000,No,2024-04-08,120,Delivered,Normal,OfficeKart,Bengaluru,Stationery,415,13200,UPI,Paid,13200
308,206,117,Mixer Grinder,HOME APPLIANCES,Kolkata,3500,45,Medium Stock,No,157500,No,2024-04-04,7,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,408,24500,Debit Card,Paid,24500
309,210,118,Water Purifier,HOME APPLIANCES,Delhi,12000,20,Medium Stock,Yes,240000,No,2024-04-05,4,Pending,High,Kitchen World,Chennai,Home Appliances,409,48000,UPI,Pending,48000
310,206,119,Ceiling Fan,HOME APPLIANCES,Ahmedabad,2800,60,Medium Stock,No,168000,No,2024-04-05,12,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,410,33600,Cash,Paid,33600
303,204,111,LED TV,ELECTRONICS,Delhi,45000,15,Low Stock,Yes,675000,Yes,2024-04-02,2,Delivered,Normal,Elite Electronics,Delhi,Electronics,403,90000,Bank Transfer,Paid,90000
304,208,114,Mobile Phone,ELECTRONICS,Hyderabad,25000,35,Medium Stock,Yes,875000,No,2024-04-02,5,Pending,High,Smart Electronics,Kolkata,Electronics,404,125000,UPI,Pending,125000
305,204,115,Laptop,ELECTRONICS,Pune,62000,18,Low Stock,Yes,1116000,Yes,2024-04-03,3,Delivered,Normal,Elite Electronics,Delhi,Electronics,405,186000,Bank Transfer,Paid,186000
318,203,108,Toothpaste,PERSONAL CARE,Ahmedabad,90,250,High Stock,No,22500,No,2024-04-09,40,Delivered,Normal,CarePlus Suppliers,Mumbai,Personal Care,418,3600,Debit Card,Paid,3600


In [0]:
latest_count = spark.table("retail_final_delta").count()
older_count = older_retail_df.count()

print("Latest count:", latest_count)
print("Older version 0 count:", older_count)

comparison_df = spark.createDataFrame(
    [("Latest Version", latest_count), ("Older Version 0", older_count)],
    ["version_type", "record_count"]
)
display(comparison_df)

Latest count: 20
Older version 0 count: 20


version_type,record_count
Latest Version,20
Older Version 0,20


In [0]:
spark.sql("VACUUM retail_final_delta DRY RUN").show(truncate=False)

+----+
|path|
+----+
+----+



## Part 8 — Merge / Upsert

In [0]:
daily_orders_data = [
    (321,114,204,"2024-04-11",3,"Delivered"),
    (322,118,210,"2024-04-11",2,"Delivered"),
    (304,114,208,"2024-04-02",5,"Delivered"),
    (319,112,208,"2024-04-10",2,"Delivered"),
    (323,120,210,"2024-04-12",1,"Pending")
]

daily_orders_columns = ["order_id","product_id","supplier_id","order_date","quantity","order_status"]

daily_orders_df = spark.createDataFrame(daily_orders_data, daily_orders_columns) \
    .withColumn("order_date", to_date(col("order_date")))

display(daily_orders_df)

order_id,product_id,supplier_id,order_date,quantity,order_status
321,114,204,2024-04-11,3,Delivered
322,118,210,2024-04-11,2,Delivered
304,114,208,2024-04-02,5,Delivered
319,112,208,2024-04-10,2,Delivered
323,120,210,2024-04-12,1,Pending


In [0]:
spark.sql("DROP TABLE IF EXISTS orders_delta_target")

DataFrame[]

In [0]:
orders_transformed_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("orders_delta_target")

display(spark.table("orders_delta_target"))

order_id,product_id,supplier_id,order_date,quantity,order_status,order_priority
301,101,201,2024-04-01,20,Delivered,Normal
302,102,201,2024-04-01,35,Delivered,Normal
303,111,204,2024-04-02,2,Delivered,Normal
304,114,208,2024-04-02,5,Pending,High
305,115,204,2024-04-03,3,Delivered,Normal
306,104,202,2024-04-03,50,Delivered,Normal
307,105,202,2024-04-04,18,Cancelled,Low
308,117,206,2024-04-04,7,Delivered,Normal
309,118,210,2024-04-05,4,Pending,High
310,119,206,2024-04-05,12,Delivered,Normal


In [0]:
daily_orders_df.createOrReplaceTempView("daily_orders_updates")

In [0]:
spark.sql("""
MERGE INTO orders_delta_target AS target
USING daily_orders_updates AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
  UPDATE SET
    target.product_id = source.product_id,
    target.supplier_id = source.supplier_id,
    target.order_date = source.order_date,
    target.quantity = source.quantity,
    target.order_status = source.order_status

WHEN NOT MATCHED THEN
  INSERT (
    order_id,
    product_id,
    supplier_id,
    order_date,
    quantity,
    order_status
  )
  VALUES (
    source.order_id,
    source.product_id,
    source.supplier_id,
    source.order_date,
    source.quantity,
    source.order_status
  )
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(spark.sql("SELECT * FROM orders_delta_target WHERE order_id IN (304, 319) ORDER BY order_id"))

order_id,product_id,supplier_id,order_date,quantity,order_status,order_priority
304,114,208,2024-04-02,5,Delivered,High
319,112,208,2024-04-10,2,Delivered,High


In [0]:
display(spark.sql("SELECT * FROM orders_delta_target WHERE order_id IN (321, 322, 323) ORDER BY order_id"))

order_id,product_id,supplier_id,order_date,quantity,order_status,order_priority
321,114,204,2024-04-11,3,Delivered,null
322,118,210,2024-04-11,2,Delivered,null
323,120,210,2024-04-12,1,Pending,null


In [0]:
display(spark.sql("DESCRIBE HISTORY orders_delta_target"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-05-06T10:49:57.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(order_id#37152L = order_id#36707L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2392048196024932),3713f110-4590-454f-af1f-757d34bb669f,0506-100524-iqf6z1vx-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 5, numTargetBytesAdded -> 10054, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2378, materializeSourceTimeMs -> 69, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1050, numTargetRowsUpdated -> 2, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1233)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-06T10:49:10.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2392048196024932),04585a4d-21f8-4cd6-b493-89c63ffcb0a0,0506-100524-iqf6z1vx-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 20, numOutputBytes -> 2521)",null,Databricks-Runtime/18.1.x-photon-scala2.13


In [0]:
display(spark.sql("DESCRIBE HISTORY orders_delta_target"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-05-06T10:49:57.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(order_id#37152L = order_id#36707L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2392048196024932),3713f110-4590-454f-af1f-757d34bb669f,0506-100524-iqf6z1vx-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 5, numTargetBytesAdded -> 10054, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 2378, materializeSourceTimeMs -> 69, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1050, numTargetRowsUpdated -> 2, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1233)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-06T10:49:10.000Z,147185198727538,azuser6402_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2392048196024932),04585a4d-21f8-4cd6-b493-89c63ffcb0a0,0506-100524-iqf6z1vx-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 20, numOutputBytes -> 2521)",null,Databricks-Runtime/18.1.x-photon-scala2.13


**Why MERGE is important?**

MERGE is important because daily incoming data can contain both existing records and new records.
If an order already exists, MERGE updates it with the latest values.
If an order does not exist, MERGE inserts it as a new row.
This avoids reloading the entire dataset again and makes incremental data loading faster and more efficient.

## Part 9 — Parquet to Delta

In [0]:
spark.sql("DROP TABLE IF EXISTS products_parquet_table")
products_transformed_df.write.format("delta").mode("overwrite").saveAsTable("products_parquet_table")

In [0]:
products_parquet_df = spark.table("products_parquet_table")
display(products_parquet_df)

product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag
101,Rice Bag,GROCERIES,Hyderabad,1200,50,Medium Stock,No,60000,No
102,Wheat Flour,GROCERIES,Bengaluru,900,80,Medium Stock,No,72000,No
103,Sunflower Oil,GROCERIES,Mumbai,1800,40,Medium Stock,No,72000,No
104,Milk Pack,DAIRY,Chennai,60,200,High Stock,No,12000,No
105,Cheese Block,DAIRY,Delhi,450,70,Medium Stock,No,31500,No
106,Soap,PERSONAL CARE,Kolkata,120,300,High Stock,No,36000,No
107,Shampoo,PERSONAL CARE,Pune,320,150,High Stock,No,48000,No
108,Toothpaste,PERSONAL CARE,Ahmedabad,90,250,High Stock,No,22500,No
109,Notebook,STATIONERY,Hyderabad,75,500,High Stock,No,37500,No
110,Pen Pack,STATIONERY,Mumbai,110,400,High Stock,No,44000,No


In [0]:
spark.sql("DROP TABLE IF EXISTS products_delta_table")
products_parquet_df.write.format("delta").mode("overwrite").saveAsTable("products_delta_table")

products_delta_df = spark.table("products_delta_table")

display(products_delta_df)

product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag
101,Rice Bag,GROCERIES,Hyderabad,1200,50,Medium Stock,No,60000,No
102,Wheat Flour,GROCERIES,Bengaluru,900,80,Medium Stock,No,72000,No
103,Sunflower Oil,GROCERIES,Mumbai,1800,40,Medium Stock,No,72000,No
104,Milk Pack,DAIRY,Chennai,60,200,High Stock,No,12000,No
105,Cheese Block,DAIRY,Delhi,450,70,Medium Stock,No,31500,No
106,Soap,PERSONAL CARE,Kolkata,120,300,High Stock,No,36000,No
107,Shampoo,PERSONAL CARE,Pune,320,150,High Stock,No,48000,No
108,Toothpaste,PERSONAL CARE,Ahmedabad,90,250,High Stock,No,22500,No
109,Notebook,STATIONERY,Hyderabad,75,500,High Stock,No,37500,No
110,Pen Pack,STATIONERY,Mumbai,110,400,High Stock,No,44000,No


In [0]:
print("Source table count:", products_parquet_df.count())
print("Delta table count:", products_delta_df.count())

display(spark.table("products_delta_table"))

Source table count: 20
Delta table count: 20


product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag
101,Rice Bag,GROCERIES,Hyderabad,1200,50,Medium Stock,No,60000,No
102,Wheat Flour,GROCERIES,Bengaluru,900,80,Medium Stock,No,72000,No
103,Sunflower Oil,GROCERIES,Mumbai,1800,40,Medium Stock,No,72000,No
104,Milk Pack,DAIRY,Chennai,60,200,High Stock,No,12000,No
105,Cheese Block,DAIRY,Delhi,450,70,Medium Stock,No,31500,No
106,Soap,PERSONAL CARE,Kolkata,120,300,High Stock,No,36000,No
107,Shampoo,PERSONAL CARE,Pune,320,150,High Stock,No,48000,No
108,Toothpaste,PERSONAL CARE,Ahmedabad,90,250,High Stock,No,22500,No
109,Notebook,STATIONERY,Hyderabad,75,500,High Stock,No,37500,No
110,Pen Pack,STATIONERY,Mumbai,110,400,High Stock,No,44000,No


**Compare Parquet vs Delta behavior**

Parquet is a storage file format used for fast reading and compression.
Delta Lake is built on top of Parquet and supports extra features like UPDATE, DELETE, MERGE, transaction history, time travel and ACID transactions.
Parquet is good for storing stable data, but Delta is better for data engineering pipelines where records can change frequently.

In [0]:
spark.sql("""
UPDATE products_delta_table
SET stock_quantity = 100
WHERE product_id = 116
""")

display(spark.sql("SELECT product_id, product_name, stock_quantity FROM products_delta_table WHERE product_id = 116"))

product_id,product_name,stock_quantity
116,Air Conditioner,100


**Why Delta supports updates better?**

Delta supports updates better because it allows direct row-level UPDATE commands.
Raw Parquet files cannot be updated directly like a database table.
Usually, we need to read the data, modify it, and rewrite the files again.
Delta uses a transaction log to safely track changes, so updates are easier, safer, and more reliable.

## Part 11 — Unity Catalog and Governance

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS retail_supply_chain_catalog")

spark.sql("CREATE SCHEMA IF NOT EXISTS retail_supply_chain_catalog.retail_governance")

spark.sql("""
CREATE OR REPLACE TABLE retail_supply_chain_catalog.retail_governance.retail_final_uc
USING DELTA
AS
SELECT *
FROM retail_final_delta
""")

display(spark.sql("SELECT * FROM retail_supply_chain_catalog.retail_governance.retail_final_uc"))

spark.sql("""
CREATE OR REPLACE TABLE retail_supply_chain_catalog.retail_governance.category_revenue_summary
USING DELTA
AS
SELECT 
    category,
    SUM(bill_amount) AS total_revenue,
    COUNT(order_id) AS total_orders
FROM retail_supply_chain_catalog.retail_governance.retail_final_uc
GROUP BY category
""")

display(spark.sql("SELECT * FROM retail_supply_chain_catalog.retail_governance.category_revenue_summary"))


# 100. Apply SELECT permissions and explain governance behavior


# Unity Catalog helps manage data governance by controlling who can access catalogs, schemas, tables and views.
# SELECT permission allows users to read the table but not modify it.
# Lineage helps us understand how one table is created from another table, which improves tracking, auditing and data trust.


order_id,supplier_id,product_id,product_name,category,inventory_city,price,stock_quantity,stock_status,expensive_product_flag,inventory_value,low_stock_flag,order_date,quantity,order_status,order_priority,supplier_name,supplier_city,specialization,payment_id,bill_amount,payment_mode,payment_status,total_order_value
301,201,101,Rice Bag,GROCERIES,Hyderabad,1200,50,Medium Stock,No,60000,No,2024-04-01,20,Delivered,Normal,Reddy Traders,Hyderabad,Groceries,401,24000,UPI,Paid,24000
302,201,102,Wheat Flour,GROCERIES,Bengaluru,900,80,Medium Stock,No,72000,No,2024-04-01,35,Delivered,Normal,Reddy Traders,Hyderabad,Groceries,402,31500,Credit Card,Paid,31500
303,204,111,LED TV,ELECTRONICS,Delhi,45000,15,Low Stock,Yes,675000,Yes,2024-04-02,2,Delivered,Normal,Elite Electronics,Delhi,Electronics,403,90000,Bank Transfer,Paid,90000
304,208,114,Mobile Phone,ELECTRONICS,Hyderabad,25000,35,Medium Stock,Yes,875000,No,2024-04-02,5,Pending,High,Smart Electronics,Kolkata,Electronics,404,125000,UPI,Pending,125000
305,204,115,Laptop,ELECTRONICS,Pune,62000,18,Low Stock,Yes,1116000,Yes,2024-04-03,3,Delivered,Normal,Elite Electronics,Delhi,Electronics,405,186000,Bank Transfer,Paid,186000
306,202,104,Milk Pack,DAIRY,Chennai,60,200,High Stock,No,12000,No,2024-04-03,50,Delivered,Normal,Fresh Dairy Ltd,Chennai,Dairy,406,3000,Cash,Paid,3000
308,206,117,Mixer Grinder,HOME APPLIANCES,Kolkata,3500,45,Medium Stock,No,157500,No,2024-04-04,7,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,408,24500,Debit Card,Paid,24500
309,210,118,Water Purifier,HOME APPLIANCES,Delhi,12000,20,Medium Stock,Yes,240000,No,2024-04-05,4,Pending,High,Kitchen World,Chennai,Home Appliances,409,48000,UPI,Pending,48000
310,206,119,Ceiling Fan,HOME APPLIANCES,Ahmedabad,2800,60,Medium Stock,No,168000,No,2024-04-05,12,Delivered,Normal,HomeNeeds Pvt Ltd,Pune,Home Appliances,410,33600,Cash,Paid,33600
311,210,120,Gas Stove,HOME APPLIANCES,Chennai,5500,25,Medium Stock,No,137500,No,2024-04-06,6,Delivered,Normal,Kitchen World,Chennai,Home Appliances,411,33000,Credit Card,Paid,33000


category,total_revenue,total_orders
HOME APPLIANCES,163100,5
GROCERIES,73500,3
STATIONERY,19200,2
DAIRY,3000,1
ELECTRONICS,752000,7
PERSONAL CARE,11600,2
